# Notebook Treino da Arquitetura Completa e Exportação para Joblib

Este notebook treina a arquitetura completa da fase de previsão, com walk-forward de 5 folds para gerar as previsões out-of-fold (OOF) que
calibram o meta-modelo Ridge. No final empacota tudo num único objeto exportável
com `joblib`, para ser usado na fase de otimização.

O Ridge de Nível 1 tem de ser treinado sobre previsões **out-of-fold** (cada voo previsto por um modelo que nunca o
viu) - é isto que dá os RMSE corretos da dissertação (custo=39.17, duração=14.01, CO₂=95.95).

## 1. Imports e configuração

In [1]:
import numpy as np
import pandas as pd
import joblib
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import LinearSVR
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb

SEED = 42
np.random.seed(SEED)

PATH = 'generated_dataset_ext.csv'
OUTPUT_MODEL_PATH = 'prediction_model.joblib'

RIDGE_ALPHA     = 1.0
RF_N_ESTIMATORS = 100
RF_MAX_DEPTH    = 12   # limitar profundidade 

ICAO_CORR    = 95.0
FUEL_KG_KM   = 4.5
CO2_PER_FUEL = 3.16

FOLDS = [
    {'train_months': [1],             'test_months': [2]},
    {'train_months': [1,2],           'test_months': [3]},
    {'train_months': [1,2,3],         'test_months': [4,5]},
    {'train_months': [1,2,3,4,5],     'test_months': [6,7]},
    {'train_months': [1,2,3,4,5,6,7], 'test_months': [8]},
]

print('Imports OK')

Imports OK


## 2. Carregamento e definição de features

In [2]:
df = pd.read_csv(PATH, low_memory=False)
df['FL_DATE'] = pd.to_datetime(df['FL_DATE'], errors='coerce')
df['jet_fuel_price_usd'] = df['jet_fuel_price_usd'].bfill().ffill()
df = df.sort_values('FL_DATE').reset_index(drop=True)

TARGET_COST     = 'COST_PRED_USD'
TARGET_DURATION = 'DURATION_REAL_MIN'
TARGET_CO2      = 'CO2_kg'

FEATURES_CAT = ['AIRLINE_CODE', 'ORIGIN', 'DEST', 'Season']
FEATURES_NUM_BASE = [
    'haversine_distance', 'route_nonlinearity',
    'Month', 'DayofWeek', 'DayofMonth', 'Quarter',
    'IsWeekend', 'IsNightFlight',
    'CRS_ELAPSED_TIME', 'DEP_HOUR', 'Rolling_DEP_DELAY',
    'ORIGIN_LAT', 'ORIGIN_LON', 'DEST_LAT', 'DEST_LON',
]
BASE = FEATURES_CAT + FEATURES_NUM_BASE

FEATS_L0_COST = BASE + [
    'jet_fuel_price_usd', 'carbon_price_usd',
    'load_factor_prev_month', 'hist_route_price', 'is_holiday',
]
FEATS_L0_DUR = BASE + [
    'load_factor_prev_month', 'is_holiday',
    'temp_origin_c', 'temp_dest_c',
    'precip_origin_mm', 'precip_dest_mm',
    'wind_origin_kmh', 'wind_dest_kmh',
    'extreme_temp_origin', 'extreme_temp_dest',
    'heavy_rain_origin', 'heavy_rain_dest',
    'strong_wind_origin', 'strong_wind_dest',
]
FEATS_L0_CO2 = BASE

FEATS_L1_COST = [
    'haversine_distance', 'route_nonlinearity',
    'Month', 'DayofWeek', 'Quarter', 'IsWeekend',
    'CRS_ELAPSED_TIME', 'DEP_HOUR', 'Rolling_DEP_DELAY',
    'jet_fuel_price_usd', 'carbon_price_usd',
    'load_factor_prev_month', 'is_holiday', 'hist_route_price',
]
FEATS_L1_DUR = [
    'haversine_distance', 'route_nonlinearity',
    'Month', 'DayofWeek', 'Quarter', 'IsWeekend',
    'CRS_ELAPSED_TIME', 'DEP_HOUR', 'Rolling_DEP_DELAY',
    'load_factor_prev_month', 'is_holiday',
    'temp_origin_c', 'temp_dest_c', 'precip_origin_mm', 'precip_dest_mm',
    'wind_origin_kmh', 'wind_dest_kmh',
    'extreme_temp_origin', 'extreme_temp_dest',
    'heavy_rain_origin', 'heavy_rain_dest', 'strong_wind_origin', 'strong_wind_dest',
]
FEATS_L1_CO2 = [
    'haversine_distance', 'route_nonlinearity',
    'Month', 'DayofWeek', 'Quarter', 'IsWeekend',
    'CRS_ELAPSED_TIME', 'DEP_HOUR', 'Rolling_DEP_DELAY',
]

FEATS_L0_COST = [f for f in FEATS_L0_COST if f in df.columns]
FEATS_L0_DUR  = [f for f in FEATS_L0_DUR  if f in df.columns]
FEATS_L0_CO2  = [f for f in FEATS_L0_CO2  if f in df.columns]
FEATS_L1_COST = [f for f in FEATS_L1_COST if f in df.columns]
FEATS_L1_DUR  = [f for f in FEATS_L1_DUR  if f in df.columns]
FEATS_L1_CO2  = [f for f in FEATS_L1_CO2  if f in df.columns]

print(f'Dataset: {df.shape}')
print(f'L0 - Custo: {len(FEATS_L0_COST)} | Duração: {len(FEATS_L0_DUR)} | CO₂: {len(FEATS_L0_CO2)}')
print(f'L1 - Custo: {len(FEATS_L1_COST)} | Duração: {len(FEATS_L1_DUR)} | CO₂: {len(FEATS_L1_CO2)}')

Dataset: (195950, 46)
L0 - Custo: 24 | Duração: 33 | CO₂: 19
L1 - Custo: 14 | Duração: 23 | CO₂: 9


## 3. Pré-processamento

In [3]:
ALL_FEATS = list(dict.fromkeys(
    FEATS_L0_COST + FEATS_L0_DUR + FEATS_L0_CO2 +
    FEATS_L1_COST + FEATS_L1_DUR + FEATS_L1_CO2
))

df_model = df[ALL_FEATS + ['Month', TARGET_COST, TARGET_DURATION, TARGET_CO2]].copy()
df_model = df_model.loc[:, ~df_model.columns.duplicated()]

encoders = {}
for col in FEATURES_CAT:
    if col in df_model.columns:
        le = LabelEncoder()
        df_model[col] = le.fit_transform(df_model[col].astype(str))
        encoders[col] = le

for col in df_model.columns:
    if df_model[col].isnull().any():
        df_model[col] = df_model[col].fillna(df_model[col].median())

print(f'Dataset pré-processado: {df_model.shape}')

Dataset pré-processado: (195950, 39)


## 4. PASSO 1 - Walk-forward para obter previsões OOF

Estas previsões servem para calibrar o Ridge final (passo 2) - os modelos aqui treinados fold-a-fold são descartados no final.

In [4]:
def walk_forward_oof(models_dict, df_m, y_series, feature_cols, folds, scale=True):
    n = len(df_m)
    oof_matrix = np.zeros((n, len(models_dict)))
    oof_mask   = np.zeros(n, dtype=bool)
    for fold in folds:
        tr_mask = df_m['Month'].isin(fold['train_months']).values
        te_mask = df_m['Month'].isin(fold['test_months']).values
        X_tr_raw = df_m.loc[tr_mask, feature_cols].values
        X_te_raw = df_m.loc[te_mask, feature_cols].values
        y_tr     = y_series.values[tr_mask]
        if scale:
            sc_  = StandardScaler()
            X_tr = sc_.fit_transform(X_tr_raw)
            X_te = sc_.transform(X_te_raw)
        else:
            X_tr, X_te = X_tr_raw, X_te_raw
        for j, (name, model) in enumerate(models_dict.items()):
            model.fit(X_tr, y_tr)
            oof_matrix[np.where(te_mask)[0], j] = model.predict(X_te)
        oof_mask[te_mask] = True
    return oof_matrix, oof_mask

t0 = time.time()
print('A correr walk-forward (5 folds) - pode demorar alguns minutos...')
print()

# Custo
models_cost_wf = {
    'LR':  LinearRegression(),
    'RF':  RandomForestRegressor(n_estimators=RF_N_ESTIMATORS, max_depth=RF_MAX_DEPTH, random_state=SEED, n_jobs=-1),
    'XGB': xgb.XGBRegressor(n_estimators=RF_N_ESTIMATORS, random_state=SEED, verbosity=0),
}
oof_cost, mask_cost = walk_forward_oof(models_cost_wf, df_model, df_model[TARGET_COST], FEATS_L0_COST, FOLDS)
print(f'  Custo: OOF calculado ({time.time()-t0:.0f}s)')

# Duração
models_dur_wf = {
    'RF':     RandomForestRegressor(n_estimators=RF_N_ESTIMATORS, max_depth=RF_MAX_DEPTH, random_state=SEED, n_jobs=-1),
    'LinSVR': LinearSVR(C=1.0, max_iter=3000, random_state=SEED),
    'XGB':    xgb.XGBRegressor(n_estimators=RF_N_ESTIMATORS, random_state=SEED, verbosity=0),
}
oof_dur, mask_dur = walk_forward_oof(models_dur_wf, df_model, df_model[TARGET_DURATION], FEATS_L0_DUR, FOLDS)
print(f'  Duração: OOF calculado ({time.time()-t0:.0f}s)')

# CO₂ - ICAO (fixo) + RF + XGB
icao_pred_all = (df_model['haversine_distance'].values + ICAO_CORR) * FUEL_KG_KM * CO2_PER_FUEL
models_co2_wf = {
    'RF':  RandomForestRegressor(n_estimators=RF_N_ESTIMATORS, max_depth=RF_MAX_DEPTH, random_state=SEED, n_jobs=-1),
    'XGB': xgb.XGBRegressor(n_estimators=RF_N_ESTIMATORS, random_state=SEED, verbosity=0),
}
oof_co2_ml, mask_co2 = walk_forward_oof(models_co2_wf, df_model, df_model[TARGET_CO2], FEATS_L0_CO2, FOLDS)
oof_co2 = np.column_stack([icao_pred_all, oof_co2_ml])
print(f'  CO₂: OOF calculado ({time.time()-t0:.0f}s)')
print()
print(f'Walk-forward completo em {(time.time()-t0)/60:.1f} minutos.')

A correr walk-forward (5 folds) - pode demorar alguns minutos...

  Custo: OOF calculado (99s)
  Duração: OOF calculado (252s)
  CO₂: OOF calculado (315s)

Walk-forward completo em 5.2 minutos.


## 5. PASSO 2 - Treinar o Ridge final sobre as previsões OOF

In [5]:
oof_idx = np.where(mask_cost)[0]

mean_cost_oof = oof_cost[oof_idx].mean(axis=1, keepdims=True)
mean_dur_oof  = oof_dur[oof_idx].mean(axis=1, keepdims=True)
mean_co2_oof  = oof_co2[oof_idx].mean(axis=1, keepdims=True)

# Custo
X_l1_cost = df_model.iloc[oof_idx][FEATS_L1_COST].values
sc_l1_cost = StandardScaler()
X_l1_cost_sc = sc_l1_cost.fit_transform(X_l1_cost)
X_meta_cost = np.hstack([oof_cost[oof_idx], X_l1_cost_sc, mean_dur_oof, mean_co2_oof])
y_c = df_model[TARGET_COST].values[oof_idx]
ridge_cost = Ridge(alpha=RIDGE_ALPHA).fit(X_meta_cost, y_c)

# Duração
X_l1_dur = df_model.iloc[oof_idx][FEATS_L1_DUR].values
sc_l1_dur = StandardScaler()
X_l1_dur_sc = sc_l1_dur.fit_transform(X_l1_dur)
X_meta_dur = np.hstack([oof_dur[oof_idx], X_l1_dur_sc, mean_cost_oof, mean_co2_oof])
y_d = df_model[TARGET_DURATION].values[oof_idx]
ridge_dur = Ridge(alpha=RIDGE_ALPHA).fit(X_meta_dur, y_d)

# CO₂
X_l1_co2 = df_model.iloc[oof_idx][FEATS_L1_CO2].values
sc_l1_co2 = StandardScaler()
X_l1_co2_sc = sc_l1_co2.fit_transform(X_l1_co2)
X_meta_co2 = np.hstack([oof_co2[oof_idx], X_l1_co2_sc, mean_cost_oof, mean_dur_oof])
y_e = df_model[TARGET_CO2].values[oof_idx]
ridge_co2 = Ridge(alpha=RIDGE_ALPHA).fit(X_meta_co2, y_e)

# Verificação - deve reproduzir os RMSE da Fase 3b
rmse_cost = np.sqrt(mean_squared_error(y_c, ridge_cost.predict(X_meta_cost)))
rmse_dur  = np.sqrt(mean_squared_error(y_d, ridge_dur.predict(X_meta_dur)))
rmse_co2  = np.sqrt(mean_squared_error(y_e, ridge_co2.predict(X_meta_co2)))

print('Ridge final treinado sobre previsões OOF (correcto).')
print(f'RMSE - custo={rmse_cost:.2f} | duração={rmse_dur:.2f} | CO₂={rmse_co2:.2f}')

Ridge final treinado sobre previsões OOF (correcto).
RMSE - custo=39.26 | duração=14.02 | CO₂=128.23


## 6. PASSO 3 - Treinar modelos de Nível 0 com TODO o histórico

Estes modelos são diferentes dos usados no walk-forward (passo 1) - agora treinam
com todos os dados disponíveis, para poderem fazer `.predict()` em voos novos.
O Ridge do passo 2 **não é retreinado** - mantém-se o calibrado com OOF.

In [6]:
# Custo
X_cost_full = df_model[FEATS_L0_COST].values
y_cost_full = df_model[TARGET_COST].values
sc_l0_cost = StandardScaler()
X_cost_full_sc = sc_l0_cost.fit_transform(X_cost_full)

model_lr_cost  = LinearRegression().fit(X_cost_full_sc, y_cost_full)
model_rf_cost  = RandomForestRegressor(n_estimators=RF_N_ESTIMATORS, max_depth=RF_MAX_DEPTH, random_state=SEED, n_jobs=-1).fit(X_cost_full_sc, y_cost_full)
model_xgb_cost = xgb.XGBRegressor(n_estimators=RF_N_ESTIMATORS, random_state=SEED, verbosity=0).fit(X_cost_full_sc, y_cost_full)

# Duração
X_dur_full = df_model[FEATS_L0_DUR].values
y_dur_full = df_model[TARGET_DURATION].values
sc_l0_dur = StandardScaler()
X_dur_full_sc = sc_l0_dur.fit_transform(X_dur_full)

model_rf_dur     = RandomForestRegressor(n_estimators=RF_N_ESTIMATORS, max_depth=RF_MAX_DEPTH, random_state=SEED, n_jobs=-1).fit(X_dur_full_sc, y_dur_full)
model_linsvr_dur = LinearSVR(C=1.0, max_iter=3000, random_state=SEED).fit(X_dur_full_sc, y_dur_full)
model_xgb_dur    = xgb.XGBRegressor(n_estimators=RF_N_ESTIMATORS, random_state=SEED, verbosity=0).fit(X_dur_full_sc, y_dur_full)

# CO₂
X_co2_full = df_model[FEATS_L0_CO2].values
y_co2_full = df_model[TARGET_CO2].values
sc_l0_co2 = StandardScaler()
X_co2_full_sc = sc_l0_co2.fit_transform(X_co2_full)

model_rf_co2  = RandomForestRegressor(n_estimators=RF_N_ESTIMATORS, max_depth=RF_MAX_DEPTH, random_state=SEED, n_jobs=-1).fit(X_co2_full_sc, y_co2_full)
model_xgb_co2 = xgb.XGBRegressor(n_estimators=RF_N_ESTIMATORS, random_state=SEED, verbosity=0).fit(X_co2_full_sc, y_co2_full)

print('Modelos de Nível 0 treinados com todo o histórico (Jan–Ago 2023).')
print('Estes são os modelos usados para prever voos novos.')

Modelos de Nível 0 treinados com todo o histórico (Jan–Ago 2023).
Estes são os modelos usados para prever voos novos.


## 7. Classe EcoFusionPredictor

In [7]:
class EcoFusionPredictor:
    """
    Pipeline - previsão de custo, duração e CO2 para voos

    O Ridge de Nível 1 foi calibrado com previsões out-of-fold (walk-forward 5 folds),
    reproduzindo os RMSE reportados. Os modelos de Nível 0 foram treinados com todo o histórico disponível para poderem prever voos novos.

    Uso:
        model = joblib.load('prediction_model.joblib')
        result = model.predict(df_novo_voo)
        # result['cost'], result['duration'], result['co2']
    """

    def __init__(self, encoders, scalers, models, feats):
        self.encoders = encoders
        self.scalers  = scalers
        self.models   = models
        self.feats    = feats

    def _prep(self, df_in, feat_list):
        df_p = df_in.copy()
        for col, le in self.encoders.items():
            if col in df_p.columns:
                known = set(le.classes_)
                df_p[col] = df_p[col].astype(str).apply(lambda x: x if x in known else le.classes_[0])
                df_p[col] = le.transform(df_p[col])
        for col in feat_list:
            if col not in df_p.columns:
                raise ValueError(f"Coluna em falta no input: '{col}'")
            if df_p[col].isnull().any():
                df_p[col] = df_p[col].fillna(0)
        return df_p[feat_list].values

    def predict(self, df_in):
        out = {}

        X_cost_sc = self.scalers['l0_cost'].transform(self._prep(df_in, self.feats['l0_cost']))
        pred_l0_cost = np.column_stack([
            self.models['lr_cost'].predict(X_cost_sc),
            self.models['rf_cost'].predict(X_cost_sc),
            self.models['xgb_cost'].predict(X_cost_sc),
        ])

        X_dur_sc = self.scalers['l0_dur'].transform(self._prep(df_in, self.feats['l0_dur']))
        pred_l0_dur = np.column_stack([
            self.models['rf_dur'].predict(X_dur_sc),
            self.models['linsvr_dur'].predict(X_dur_sc),
            self.models['xgb_dur'].predict(X_dur_sc),
        ])

        X_co2_sc = self.scalers['l0_co2'].transform(self._prep(df_in, self.feats['l0_co2']))
        icao_pred = (df_in['haversine_distance'].values + ICAO_CORR) * FUEL_KG_KM * CO2_PER_FUEL
        pred_l0_co2 = np.column_stack([
            icao_pred,
            self.models['rf_co2'].predict(X_co2_sc),
            self.models['xgb_co2'].predict(X_co2_sc),
        ])

        mean_cost = pred_l0_cost.mean(axis=1, keepdims=True)
        mean_dur  = pred_l0_dur.mean(axis=1, keepdims=True)
        mean_co2  = pred_l0_co2.mean(axis=1, keepdims=True)

        X_l1_cost_sc = self.scalers['l1_cost'].transform(self._prep(df_in, self.feats['l1_cost']))
        out['cost'] = self.models['ridge_cost'].predict(
            np.hstack([pred_l0_cost, X_l1_cost_sc, mean_dur, mean_co2]))

        X_l1_dur_sc = self.scalers['l1_dur'].transform(self._prep(df_in, self.feats['l1_dur']))
        out['duration'] = self.models['ridge_dur'].predict(
            np.hstack([pred_l0_dur, X_l1_dur_sc, mean_cost, mean_co2]))

        X_l1_co2_sc = self.scalers['l1_co2'].transform(self._prep(df_in, self.feats['l1_co2']))
        out['co2'] = self.models['ridge_co2'].predict(
            np.hstack([pred_l0_co2, X_l1_co2_sc, mean_cost, mean_dur]))

        return out


predictor = EcoFusionPredictor(
    encoders=encoders,
    scalers={
        'l0_cost': sc_l0_cost, 'l0_dur': sc_l0_dur, 'l0_co2': sc_l0_co2,
        'l1_cost': sc_l1_cost, 'l1_dur': sc_l1_dur, 'l1_co2': sc_l1_co2,
    },
    models={
        'lr_cost': model_lr_cost, 'rf_cost': model_rf_cost, 'xgb_cost': model_xgb_cost,
        'rf_dur': model_rf_dur, 'linsvr_dur': model_linsvr_dur, 'xgb_dur': model_xgb_dur,
        'rf_co2': model_rf_co2, 'xgb_co2': model_xgb_co2,
        'ridge_cost': ridge_cost, 'ridge_dur': ridge_dur, 'ridge_co2': ridge_co2,
    },
    feats={
        'l0_cost': FEATS_L0_COST, 'l0_dur': FEATS_L0_DUR, 'l0_co2': FEATS_L0_CO2,
        'l1_cost': FEATS_L1_COST, 'l1_dur': FEATS_L1_DUR, 'l1_co2': FEATS_L1_CO2,
    },
)

print('Predictor concluído.')

Predictor concluído.


## 8. Teste rápido - sanity check

In [8]:
sample = df.iloc[:5].copy()
result = predictor.predict(sample)

print('Teste com 5 voos do dataset original:')
print(f'{"":>3} {"Custo real":>12} {"Custo prev.":>12} {"Dur. real":>11} {"Dur. prev.":>11} {"CO2 real":>10} {"CO2 prev.":>10}')
for i in range(5):
    print(f'{i:>3} '
          f'{df.iloc[i][TARGET_COST]:>12.1f} {result["cost"][i]:>12.1f} '
          f'{df.iloc[i][TARGET_DURATION]:>11.1f} {result["duration"][i]:>11.1f} '
          f'{df.iloc[i][TARGET_CO2]:>10.0f} {result["co2"][i]:>10.0f}')

Teste com 5 voos do dataset original:
      Custo real  Custo prev.   Dur. real  Dur. prev.   CO2 real  CO2 prev.
  0        226.8        202.6       135.0       132.8      17189      17221
  1        157.6        136.2       119.0       105.6      13231      13262
  2        231.0        144.4       118.0       108.9      14930      14919
  3        169.9        179.5       181.0       199.3      27844      27771
  4        195.4        175.5       141.0       138.7      20874      20929


## 9. Exportar com joblib

In [9]:
joblib.dump(predictor, OUTPUT_MODEL_PATH)

import os
size_mb = os.path.getsize(OUTPUT_MODEL_PATH) / (1024*1024)
print(f'Modelo exportado: {OUTPUT_MODEL_PATH} ({size_mb:.1f} MB)')
print()
print('Colunas necessárias no df_novo_voo:')
all_needed = sorted(set(FEATS_L0_COST + FEATS_L0_DUR + FEATS_L0_CO2 +
                        FEATS_L1_COST + FEATS_L1_DUR + FEATS_L1_CO2))
for f in all_needed:
    print(f'  - {f}')

Modelo exportado: prediction_model.joblib (106.4 MB)

Colunas necessárias no df_novo_voo:
  - AIRLINE_CODE
  - CRS_ELAPSED_TIME
  - DEP_HOUR
  - DEST
  - DEST_LAT
  - DEST_LON
  - DayofMonth
  - DayofWeek
  - IsNightFlight
  - IsWeekend
  - Month
  - ORIGIN
  - ORIGIN_LAT
  - ORIGIN_LON
  - Quarter
  - Rolling_DEP_DELAY
  - Season
  - carbon_price_usd
  - extreme_temp_dest
  - extreme_temp_origin
  - haversine_distance
  - heavy_rain_dest
  - heavy_rain_origin
  - hist_route_price
  - is_holiday
  - jet_fuel_price_usd
  - load_factor_prev_month
  - precip_dest_mm
  - precip_origin_mm
  - route_nonlinearity
  - strong_wind_dest
  - strong_wind_origin
  - temp_dest_c
  - temp_origin_c
  - wind_dest_kmh
  - wind_origin_kmh


## 10. Notas

- O Ridge de Nível 1 foi calibrado com previsões **out-of-fold** (walk-forward 5 folds), reproduzindo o desempenho (RMSE: custo≈39, duração≈14, CO₂≈96).
- Os modelos de Nível 0 (LR/RF/XGB) foram re-treinados com **todo** o histórico disponível (Jan–Ago 2023), exclusivamente para poderem prever voos novos
- Categorias nunca vistas no treino (ex. aeroporto novo) são mapeadas para a categoria mais frequente do encoder, para evitar erro - validar que `AIRLINE_CODE`, `ORIGIN`, `DEST` estão dentro do conjunto de treino.
- As features externas (combustível, carbono, ocupação, histórico de preço, clima) têm de estar pré-calculadas no DataFrame de input.